# Phase 9: Explainability

This notebook generates global feature importance and sample property-level explanations for the current best segmented XGBoost model.

In [1]:
from pathlib import Path
import importlib
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import backend.src.config as config_module
import backend.src.explainability as explainability_module
importlib.reload(config_module)
importlib.reload(explainability_module)

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.explainability import load_trained_model, run_explainability, save_explainability_outputs

configure_logging()
ensure_directories()
PROJECT_ROOT


WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc')

In [2]:
import geopandas as gpd
import pandas as pd

model = load_trained_model(settings.models_dir / 'valuation_model.pkl')
training_gdf = gpd.read_parquet(settings.processed_data_dir / 'model_training_dataset.parquet')
predictions_df = pd.read_parquet(settings.processed_data_dir / 'valuation_predictions.parquet')
type(model), training_gdf.shape, predictions_df.shape


c:\Users\pulkit verma\miniconda3\envs\xcel\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\pulkit verma\miniconda3\envs\xcel\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\pulkit verma\miniconda3\envs\xcel\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when us

(src.model_training.SegmentedModelPipeline, (273587, 80), (54718, 85))

In [3]:
feature_importance_df, sample_explanations, summary = run_explainability(model, training_gdf, predictions_df)
summary


c:\Users\pulkit verma\miniconda3\envs\xcel\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['Adjacent_to_Metal_Road_flag']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
2026-04-30 14:41:52,839 | WARNING | src.explainability | Falling back to heuristic local explanation for segment Flat due to: X has 22 features, but OneHotEncoder is expecting 19 features as input.
c:\Users\pulkit verma\miniconda3\envs\xcel\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['Adjacent_to_Metal_Road_flag']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
2026-04-30 14:41:52,881 | WARNING | src.explainability | Falling back to heuristic local explanation for segment Flat due to: X has 22 features, but OneHotEncoder is expecting 19 features as input.


ExplainabilitySummary(model_type='SegmentedModelPipeline', segment_values=['Flat', 'Land'], total_training_rows=273587, sampled_explanations_count=5, feature_importance_rows=623, top_global_feature='GP_target_mean')

In [4]:
feature_importance_df.head(20)


,segment_value,transformed_feature,base_feature,importance
0,GLOBAL,GP_target_mean,GP_target_mean,0.077633
1,GLOBAL,Mouza_Name_target_mean,Mouza_Name_target_mean,0.077441
2,GLOBAL,property_district_Name_target_mean,property_district_Name_target_mean,0.050025
3,GLOBAL,Adjacent_to_Metal_Road_flag,Adjacent_to_Metal_Road_flag,0.049940
4,GLOBAL,property_district_Name,property_district_Name,0.031594
5,GLOBAL,presentation_year,presentation_year,0.019544
6,GLOBAL,Adjacent_to_Metal_Road,Adjacent_to_Metal_Road,0.018748
7,GLOBAL,Area,Area,0.011492
8,GLOBAL,Urban_flag,Urban_flag,0.009985
9,GLOBAL,Rural_flag,Rural_flag,0.009922


In [5]:
sample_explanations[0]


{'identifiers': {'query_year': 2018,
  'query_no': 1543860,
  'Deed_No': 7652,
  'Deed_Year': 2018,
  'sl_no_Property': 160,
  'property_district_Name': 'Howrah',
  'ps_code': 9,
  'mouza_code': 28,
  'plot_no': 245,
  'bata_plot_no': 0},
 'segment_value': 'Land',
 'explanation_method': 'xgboost_pred_contrib',
 'predicted_value_per_area': 15632448.223146215,
 'actual_value_per_area': 329012345.6790122,
 'prediction_bias_log_target': 12.033206939697266,
 'top_positive_factors': [{'base_feature': 'Transaction_code',
   'contribution_log_target': 0.9078235030174255,
   'reason': 'Transaction_code=307'},
  {'base_feature': 'Mouza_Name_target_mean',
   'contribution_log_target': 0.6167261600494385,
   'reason': 'mouza pricing context (Jangalpur)'},
  {'base_feature': 'Adjacent_to_Metal_Road_flag',
   'contribution_log_target': 0.21183747053146362,
   'reason': 'Adjacent_to_Metal_Road_flag=0.0'},
  {'base_feature': 'Area',
   'contribution_log_target': 0.19928893446922302,
   'reason': 'prop

In [6]:
feature_importance_path, sample_explanations_path, summary_path = save_explainability_outputs(
    feature_importance_df,
    sample_explanations,
    summary,
    settings.reports_dir,
)
feature_importance_path, sample_explanations_path, summary_path


2026-04-30 14:41:53,086 | INFO | src.explainability | Saved feature importance to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\feature_importance.csv
2026-04-30 14:41:53,087 | INFO | src.explainability | Saved sample property explanations to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\sample_property_explanations.json
2026-04-30 14:41:53,088 | INFO | src.explainability | Saved explainability summary to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\explainability_summary.json


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/feature_importance.csv'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/sample_property_explanations.json'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/explainability_summary.json'))

In [7]:
json.loads((settings.reports_dir / 'explainability_summary.json').read_text(encoding='utf-8'))


{'model_type': 'SegmentedModelPipeline',
 'segment_values': ['Flat', 'Land'],
 'total_training_rows': 273587,
 'sampled_explanations_count': 5,
 'feature_importance_rows': 623,
 'top_global_feature': 'GP_target_mean'}